In [ ]:
%sql
/*
  SQL Script for Data Standardization Process
  This script ensures the delivery_dt column in the f_order table is in the correct format and datatype.
  Validates and transforms delivery_dt to Decimal(38,0) if it's not in yyyymmdd format.
  Handles invalid data and prepares it for further processing.
*/

-- Setup: Ensure f_order table exists in the purgo_playground schema
CREATE TABLE IF NOT EXISTS purgo_playground.purgo_playground.f_order (
  order_nbr STRING COMMENT 'Order Number',
  order_type BIGINT COMMENT 'Order Type',
  delivery_dt DECIMAL(38,0) COMMENT 'Delivery Date in yyyymmdd format',
  order_qty DOUBLE COMMENT 'Order Quantity',
  sched_dt DECIMAL(38,0) COMMENT 'Scheduled Date in yyyymmdd format',
  expected_shipped_dt DECIMAL(38,0) COMMENT 'Expected Shipped Date in yyyymmdd format',
  actual_shipped_dt DECIMAL(38,0) COMMENT 'Actual Shipped Date in yyyymmdd format',
  order_line_nbr STRING COMMENT 'Order Line Number',
  loc_tracker_id STRING COMMENT 'Location Tracker ID',
  shipping_add STRING COMMENT 'Shipping Address',
  primary_qty DOUBLE COMMENT 'Primary Quantity',
  open_qty DOUBLE COMMENT 'Open Quantity',
  shipped_qty DOUBLE COMMENT 'Shipped Quantity',
  order_desc STRING COMMENT 'Order Description',
  flag_return STRING COMMENT 'Return Flag',
  flag_cancel STRING COMMENT 'Cancel Flag',
  cancel_dt DECIMAL(38,0) COMMENT 'Cancel Date in yyyymmdd format',
  cancel_qty DOUBLE COMMENT 'Cancel Quantity',
  crt_dt TIMESTAMP COMMENT 'Creation Date',
  updt_dt TIMESTAMP COMMENT 'Update Date'
);

-- Validation: Check if delivery_dt is in Decimal(38,0) and yyyymmdd format
WITH Validation AS (
  SELECT order_nbr, delivery_dt,
         CASE 
           WHEN LENGTH(CAST(delivery_dt AS STRING)) != 8 THEN 'INVALID_LENGTH'
           WHEN delivery_dt < 19000101 OR delivery_dt > 99991231 THEN 'INVALID_RANGE'
           WHEN delivery_dt IS NULL THEN 'NULL_VALUE'
           ELSE 'VALID'
         END AS status
  FROM purgo_playground.purgo_playground.f_order
)
-- Output results for manual inspection and debugging
SELECT order_nbr, delivery_dt, status
FROM Validation
WHERE status != 'VALID';

-- Transformation: Attempt to correct invalid delivery_dt values
UPDATE purgo_playground.purgo_playground.f_order
SET delivery_dt = TRY_CAST(CAST(delivery_dt AS STRING) AS DECIMAL(38,0))
WHERE LENGTH(CAST(delivery_dt AS STRING)) = 8
AND delivery_dt < 99991231
AND delivery_dt IS NOT NULL;

-- Cleanup: Remove records with permanently invalid delivery_dt values
DELETE FROM purgo_playground.purgo_playground.f_order
WHERE delivery_dt IS NULL
OR LENGTH(CAST(delivery_dt AS STRING)) != 8;

-- Integration: Ensure correct data is ready for further processing or integration
INSERT INTO purgo_playground.purgo_playground.supply_chain_delivery
SELECT order_nbr AS Shipment_ID, shipping_add AS Warehouse, 'Delivered' AS Delivery_Status
FROM purgo_playground.purgo_playground.f_order
WHERE LENGTH(CAST(delivery_dt AS STRING)) = 8
AND delivery_dt BETWEEN 19000101 AND 99991231;

-- End of script
